Take each zarr file produced by racmo_2km_downscaled/download_upload_to_scratch.ipynb, rechunk to 1 in he time dimension and append to one large zarr for each 

In [7]:
client.shutdown()

In [8]:
from dask.distributed import Client
client = Client()
client.cluster.scale(24)
client

Connection method: Cluster object,Cluster type: distributed.LocalCluster
Dashboard: /user/jkingslake/load%20NCs/proxy/8787/status,
Dashboard: /user/jkingslake/load%20NCs/proxy/8787/status,Workers: 4
Total threads: 4,Total memory: 14.54 GiB
Status: running,Using processes: True
Comm: tcp://127.0.0.1:46371,Workers: 4
Dashboard: /user/jkingslake/load%20NCs/proxy/8787/status,Total threads: 4
Started: Just now,Total memory: 14.54 GiB
Comm: tcp://127.0.0.1:38299,Total threads: 1
Dashboard: /user/jkingslake/load%20NCs/proxy/42637/status,Memory: 3.63 GiB
Nanny: tcp://127.0.0.1:37327,


2026-05-28 17:21:04,083 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 21f871c937f5300143bd2cfd1dea3c76 initialized by task ('rechunk-merge-rechunk-transfer-rechunk-transfer-47843cadee3e9a3af814035b3914528d', 1, 0, 0, 1, 1, 0) executed on worker tcp://127.0.0.1:32979
2026-05-28 17:21:10,064 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle ceec2017675bc48d611087377942c8f4 initialized by task ('rechunk-merge-rechunk-transfer-rechunk-transfer-47843cadee3e9a3af814035b3914528d', 14, 0, 0, 14, 0, 0) executed on worker tcp://127.0.0.1:32979
2026-05-28 17:21:11,774 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle f818bb3690dab1e1961f6f79ce017c0a initialized by task ('rechunk-merge-rechunk-transfer-rechunk-transfer-47843cadee3e9a3af814035b3914528d', 12, 0, 0, 12, 1, 0) executed on worker tcp://127.0.0.1:46511
2026-05-28 17:21:11,823 - distributed.shuffle._scheduler_plugin - WARNING - Shuffle 898423bdecd060c0f94475baf881a432 initialized by task ('rechunk-merge-

In [12]:
import xarray as xr
import pandas as pd
from tqdm import tqdm

df = pd.read_csv('record_of_zarrs.csv')

VAR_NAME = "snowmelt"  # change this
DATASET  = "snowmelt"      # change this

df_subset = df[df["dataset"] == DATASET]
f1_sorted = df_subset['zarr_path'].to_list()

OUT_PATH = f"s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/{VAR_NAME}_all_08.zarr"

# Write first file
print(f"Writing first file: {f1_sorted[0]}")
ds = xr.open_zarr(f1_sorted[0], consolidated=True, chunks={})
ds = ds.chunk(time=1, x=-1, y=-1)
for var in ds.data_vars:
    if 'chunks' in ds[var].encoding:
        del ds[var].encoding['chunks']
ds.to_zarr(OUT_PATH, mode='w', zarr_format=2)
ds.close()

# Append remaining files
for path in tqdm(f1_sorted[1:], desc="Appending files"):
    ds = xr.open_zarr(path, consolidated=True, chunks={})
    ds = ds.chunk(time=1, x=-1, y=-1)
    for var in ds.data_vars:
        if 'chunks' in ds[var].encoding:
            del ds[var].encoding['chunks']
    ds.to_zarr(OUT_PATH, mode='a', append_dim='time', zarr_format=2)
    ds.close()

print(f"Done! Written to {OUT_PATH}")

Writing first file: s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/snowmelt/snowmelt.1979_JFM.BN_RACMO2.3p2_ANT27_ERA5_3h.AIS.2km.DD.zarr


Appending files: 100%|██████████| 187/187 [58:42<00:00, 18.84s/it]

Done! Written to s3://nasa-cryo-scratch/jkingslake/Daily-2km-zarr/snowmelt_all_08.zarr
